# 🧪 Thử nghiệm: Gemini định vị chữ ký & mộc (LOADING_PLAN)

**Mục tiêu:** đưa 1 ảnh chứng từ → Gemini trả tọa độ chữ ký / mộc → cắt crop để **soát tay** + chấm tự động với **GT v4** (70 ô ký / 19 trang, nhãn người).

> ⚠️ **Thử nghiệm độc lập** — không thuộc pipeline chính, không ghi vào artifact của các tầng. Mọi output nằm trong `output/llm_sig_test/`.
> ⚠️ **Key miễn phí**: Google có thể dùng dữ liệu gửi lên → **chỉ dùng ảnh demo**, không dùng chứng từ thật của khách hàng.
> ⚠️ GT v4 là **ô cột ký** (không phải khung chặt quanh chữ ký) ⇒ chấm theo kiểu *tâm box LLM rơi vào ô nào*, không dùng IoU.

Thứ tự: Cell 1 cấu hình → Cell 2 hàm gọi API → Cell 3 **thử 1 ảnh** → Cell 4 chạy cả 19 trang (có cache) → Cell 5 cắt crop + overlay → Cell 6 chấm với GT → Cell 7 gallery soát tay.

In [ ]:
# Cell 1 — Cấu hình
import json, os, time, base64, re, io
from pathlib import Path
import cv2, numpy as np
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd()
assert (ROOT / 'AGENTS.md').exists(), 'Hãy mở notebook từ thư mục gốc D:/OCR_chuki'

def load_env(path):
    env = {}
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            env[k.strip()] = v.strip().strip('"').strip("'")
    return env

ENV = load_env(ROOT / '.env')
API_KEY = ENV.get('GEMINI_API_KEY', '')
assert API_KEY, 'Chưa có GEMINI_API_KEY trong .env'

# Đổi model ở đây để so sánh: 'gemini-flash-lite-latest' | 'gemini-flash-latest'
MODEL = ENV.get('GEMINI_MODEL', 'gemini-flash-lite-latest')

IMG_DIR  = ROOT / 'output/stage1_out/images/form_samples'   # ảnh Tầng 1 (A4, H=2200) — cùng hệ quy chiếu production
GT_PATH  = ROOT / 'output/stage4_dynamic_gt_v4/gt_labeled.json'
OUT_DIR  = ROOT / 'output/llm_sig_test' / MODEL
RAW_DIR, CROP_DIR, OVL_DIR = OUT_DIR / 'raw', OUT_DIR / 'crops', OUT_DIR / 'overlay'
for d in (RAW_DIR, CROP_DIR, OVL_DIR): d.mkdir(parents=True, exist_ok=True)

SEND_LONG_EDGE = 1600   # thu nhỏ trước khi gửi (box trả về chuẩn hóa 0-1000 nên không ảnh hưởng tọa độ)
PAD = 0.08              # nới mỗi cạnh crop 8% kích thước box
MIN_INTERVAL_S = 4.5    # giãn nhịp cho gói miễn phí (giới hạn request/phút)

GT = json.loads(GT_PATH.read_text(encoding='utf-8'))
PAGES = sorted({p['file_name'] for p in GT['pages']})
print(f'Model: {MODEL} | {len(PAGES)} trang LP | {len(GT["targets"])} ô ký GT')
print('Output:', OUT_DIR.relative_to(ROOT))

In [ ]:
# Cell 2 — Hàm gọi Gemini
import urllib.request, urllib.error

PROMPT = '''You are analysing a scanned Vietnamese delivery document (Loading Plan).
Find EVERY handwritten signature and EVERY stamp/seal (red/blue/purple ink stamp) on the page.
Do NOT include printed text, printed titles like "(Ký, ghi rõ họ tên)", table lines or logos.
A handwritten name written next to a signature belongs to the same signature box.
Return ONLY a JSON array, no prose. Each item:
{"type": "signature" | "stamp", "box_2d": [ymin, xmin, ymax, xmax], "label": "<printed role title above it, if any>", "confidence": 0.0-1.0}
Coordinates are integers normalised to 0-1000. If nothing found return [].'''

_last_call = [0.0]

def encode_image(path):
    im = Image.open(path).convert('RGB')
    s = SEND_LONG_EDGE / max(im.size)
    if s < 1: im = im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=90)
    return base64.b64encode(buf.getvalue()).decode()

def parse_boxes(text):
    m = re.search(r'\[.*\]', text, re.S)
    if not m: return None
    try: items = json.loads(m.group(0))
    except json.JSONDecodeError: return None
    out = []
    for it in items:
        b = it.get('box_2d')
        if not (isinstance(b, list) and len(b) == 4): continue
        y1, x1, y2, x2 = [max(0, min(1000, float(v))) for v in b]
        if y2 <= y1 or x2 <= x1: continue
        out.append({'type': it.get('type', '?'), 'box_norm': [y1/1000, x1/1000, y2/1000, x2/1000],
                    'label': it.get('label', ''), 'confidence': it.get('confidence')})
    return out

def call_gemini(path, model=MODEL, retries=4):
    url = f'https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent'
    body = {'contents': [{'parts': [{'inline_data': {'mime_type': 'image/jpeg', 'data': encode_image(path)}},
                                    {'text': PROMPT}]}],
            'generationConfig': {'temperature': 0, 'responseMimeType': 'application/json'}}
    for attempt in range(retries):
        wait = MIN_INTERVAL_S - (time.time() - _last_call[0])
        if wait > 0: time.sleep(wait)
        _last_call[0] = time.time()
        req = urllib.request.Request(url, data=json.dumps(body).encode(),
                                     headers={'Content-Type': 'application/json', 'x-goog-api-key': API_KEY})
        t0 = time.time()
        try:
            with urllib.request.urlopen(req, timeout=120) as r: resp = json.load(r)
            latency = time.time() - t0
            text = resp['candidates'][0]['content']['parts'][0]['text']
            return {'model': model, 'latency_s': round(latency, 2), 'raw_text': text,
                    'boxes': parse_boxes(text), 'usage': resp.get('usageMetadata')}
        except urllib.error.HTTPError as e:
            msg = e.read().decode()[:1500]
            if e.code == 429 and 'PerDay' in msg:
                # Hết quota NGÀY của gói miễn phí — thử lại vô ích, dừng luôn
                return {'model': model, 'error': f'HTTP 429 HẾT QUOTA NGÀY: {msg}', 'quota_day': True, 'boxes': None}
            if e.code in (429, 500, 503) and attempt < retries - 1:
                print(f'  HTTP {e.code} — thử lại sau {15*(attempt+1)}s'); time.sleep(15 * (attempt + 1)); continue
            return {'model': model, 'error': f'HTTP {e.code}: {msg}', 'boxes': None}
    return {'model': model, 'error': 'hết lượt thử', 'boxes': None}

def get_result(fname, force=False):
    # Có cache: chạy lại notebook không tốn thêm lượt gọi. force=True để gọi lại.
    cache = RAW_DIR / (Path(fname).stem + '.json')
    # Chỉ cache kết quả THÀNH CÔNG — lỗi (429/503...) sẽ được gọi lại ở lần chạy sau.
    if cache.exists() and not force:
        old = json.loads(cache.read_text(encoding='utf-8'))
        if not old.get('error'): return old
    res = call_gemini(IMG_DIR / fname); res['file_name'] = fname
    if not res.get('error'):
        cache.write_text(json.dumps(res, ensure_ascii=False, indent=1), encoding='utf-8')
    return res
print('OK — hàm sẵn sàng')

In [ ]:
# Cell 3 — Vẽ khung + cắt crop; THỬ 1 ẢNH
COLORS = {'signature': (0, 160, 0), 'stamp': (0, 0, 220)}

def crop_box(img, box_norm, pad=PAD):
    H, W = img.shape[:2]
    y1, x1, y2, x2 = box_norm
    dy, dx = (y2 - y1) * pad, (x2 - x1) * pad
    Y1, X1 = max(0, int((y1 - dy) * H)), max(0, int((x1 - dx) * W))
    Y2, X2 = min(H, int((y2 + dy) * H)), min(W, int((x2 + dx) * W))
    return img[Y1:Y2, X1:X2]

def draw_overlay(img, boxes, gt_targets=()):
    out = img.copy(); H, W = out.shape[:2]
    for t in gt_targets:   # ô GT: vàng = có ký (YES), xám = trống (NO)
        y1, x1, y2, x2 = t['box_norm']
        c = (0, 200, 255) if t['own_signature'] == 'YES' else (160, 160, 160)
        cv2.rectangle(out, (int(x1*W), int(y1*H)), (int(x2*W), int(y2*H)), c, 2)
    for i, b in enumerate(boxes or []):
        y1, x1, y2, x2 = b['box_norm']; c = COLORS.get(b['type'], (200, 0, 200))
        cv2.rectangle(out, (int(x1*W), int(y1*H)), (int(x2*W), int(y2*H)), c, 4)
        cv2.putText(out, f"{i}:{b['type'][:3]}", (int(x1*W), max(20, int(y1*H) - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, c, 2)
    return out

def save_outputs(res):
    img = cv2.imread(str(IMG_DIR / res['file_name'])); stem = Path(res['file_name']).stem
    gts = [t for t in GT['targets'] if t['file_name'] == res['file_name']]
    cv2.imwrite(str(OVL_DIR / f'{stem}.jpg'), draw_overlay(img, res['boxes'], gts), [cv2.IMWRITE_JPEG_QUALITY, 85])
    paths = []
    for i, b in enumerate(res['boxes'] or []):
        p = CROP_DIR / f'{stem}__{i:02d}_{b["type"]}.png'; cv2.imwrite(str(p), crop_box(img, b['box_norm'])); paths.append(p)
    return img, gts, paths

TEST_FILE = 'Load3.3__0.png'   # đổi tên file để thử ảnh khác
res = get_result(TEST_FILE)
if res.get('error'): print('❌', res['error'])
else:
    print(f"⏱ {res['latency_s']}s | {len(res['boxes'] or [])} box | usage: {res.get('usage')}")
    for i, b in enumerate(res['boxes'] or []): print(f"  {i}: {b['type']:9s} {[round(v,3) for v in b['box_norm']]} '{b['label']}' conf={b['confidence']}")
    img, gts, paths = save_outputs(res)
    plt.figure(figsize=(9, 12)); plt.imshow(cv2.cvtColor(draw_overlay(img, res['boxes'], gts), cv2.COLOR_BGR2RGB))
    plt.title('Xanh lá = chữ ký LLM · Đỏ = mộc LLM · Vàng = ô GT có ký · Xám = ô GT trống'); plt.axis('off'); plt.show()
    if paths:
        fig, ax = plt.subplots(1, len(paths), figsize=(3.2 * len(paths), 3), squeeze=False)
        for a, p in zip(ax[0], paths): a.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)); a.set_title(p.stem[-14:], fontsize=8); a.axis('off')
        plt.show()

In [ ]:
# Cell 4 — Chạy cả 19 trang (có cache trong raw/)
RESULTS = {}
t_all = time.time()
for i, f in enumerate(PAGES, 1):
    r = get_result(f); RESULTS[f] = r
    status = (r['error'][:120] if r.get('error') else f"{len(r['boxes'] or [])} box, {r.get('latency_s')}s")
    print(f'[{i:2d}/{len(PAGES)} | {100*i//len(PAGES):3d}%] {f:28s} {status}')
    if r.get('quota_day'):
        print(f'⛔ Hết quota ngày cho {MODEL} (gói miễn phí). Dừng; chạy lại cell này vào ngày mai — trang đã xong được giữ trong cache.')
        break
lat = [r['latency_s'] for r in RESULTS.values() if r.get('latency_s')]
n_err = sum(1 for r in RESULTS.values() if r.get('error') or r.get('boxes') is None)
if len(RESULTS) < len(PAGES) or n_err: print(f'⚠️ Chưa đủ {len(PAGES)} trang hợp lệ — Cell 6 sẽ chỉ chấm trên trang đã có, KHÔNG so sánh được với model khác.')
print(f'\nLỗi/không parse được: {n_err} | latency trung bình {np.mean(lat):.2f}s, max {np.max(lat):.2f}s (chỉ tính lần gọi thật, kể cả cache cũ)')

In [ ]:
# Cell 5 — Lưu crop + overlay cho mọi trang
n_crop = 0
for r in RESULTS.values():
    if r.get('boxes') is not None: n_crop += len(save_outputs(r)[2])
print(f'{n_crop} crop → {CROP_DIR.relative_to(ROOT)}\noverlay → {OVL_DIR.relative_to(ROOT)}')

In [ ]:
# Cell 6 — Chấm với GT v4 (chỉ ô chữ ký; own_signature, bỏ AMBIGUOUS)
# Luật: box 'signature' của LLM thuộc ô GT chứa TÂM box (ô cột không chồng lấn).
#   ô YES có ≥1 box → TP · ô YES không box → FN · ô NO có box → FP · ô NO không box → TN
#   box có tâm ngoài mọi ô → đếm riêng 'ngoài ô' (có thể là mộc/chữ ký ngoài khối ký — cần soát tay)
import pandas as pd
rows, outside = [], []
for f, r in RESULTS.items():
    if r.get('error') or r.get('boxes') is None: continue   # không có kết quả ≠ 'không có chữ ký'
    gts = [t for t in GT['targets'] if t['file_name'] == f]
    hits = {t['target_id']: 0 for t in gts}
    for b in (r.get('boxes') or []):
        y1, x1, y2, x2 = b['box_norm']; cy, cx = (y1 + y2) / 2, (x1 + x2) / 2
        cell = next((t for t in gts if t['box_norm'][0] <= cy <= t['box_norm'][2] and t['box_norm'][1] <= cx <= t['box_norm'][3]), None)
        if cell is None: outside.append({'file': f, 'type': b['type'], 'box': [round(v, 3) for v in b['box_norm']]})
        elif b['type'] == 'signature': hits[cell['target_id']] += 1
    for t in gts:
        if t['own_signature'] == 'AMBIGUOUS': continue
        yes, got = t['own_signature'] == 'YES', hits[t['target_id']] > 0
        rows.append({'target_id': t['target_id'], 'role': t['role'], 'required': t['required'], 'gt': t['own_signature'],
                     'llm_boxes': hits[t['target_id']], 'kq': ('TP' if got else 'FN') if yes else ('FP' if got else 'TN')})
df = pd.DataFrame(rows); c = df['kq'].value_counts().to_dict()
TP, FP, FN, TN = (c.get(k, 0) for k in ('TP', 'FP', 'FN', 'TN'))
P = TP / (TP + FP) if TP + FP else float('nan'); R = TP / (TP + FN) if TP + FN else float('nan')
n_pages_ok = sum(1 for r in RESULTS.values() if not r.get('error') and r.get('boxes') is not None)
print(f'{MODEL} ({n_pages_ok}/{len(PAGES)} trang): {len(df)} ô | TP {TP} · TN {TN} · FP {FP} · FN {FN} | Precision {P:.2%} · Recall {R:.2%}')
print(f'Box nằm ngoài mọi ô GT: {len(outside)} (không tính vào điểm)')
print('\nSo sánh tham chiếu (AGENTS.md 9.11.D, cùng GT v4 69 ô): engine ver2 44/25/0/0 · v1 44/0/25/0')
display(pd.crosstab(df['role'], df['kq']))
display(df[df['kq'].isin(['FP', 'FN'])])
(OUT_DIR / 'eval.json').write_text(json.dumps({'model': MODEL, 'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN,
    'rows': rows, 'outside': outside}, ensure_ascii=False, indent=1), encoding='utf-8')

In [ ]:
# Cell 7 — Gallery soát tay: xem overlay từng trang (đổi SHOW để lọc)
SHOW = PAGES            # ví dụ: [f for f in PAGES if f.startswith('Load_3')]
for f in SHOW:
    p = OVL_DIR / (Path(f).stem + '.jpg')
    if not p.exists(): continue
    plt.figure(figsize=(7, 9.5)); plt.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    plt.title(f, fontsize=10); plt.axis('off'); plt.show()

### Ghi chú khi đọc kết quả
- Ô **NO** trong GT = vai trò đó **chưa ký** (mực tràn từ cột bên **không** tính là ký) — đây chính là chỗ engine cũ v1 hỏng (TN = 0).
- 19 trang demo ⇒ số đo là **chỉ dấu**, không phải ước lượng tổng quát.
- Đổi `MODEL` ở Cell 1 rồi chạy lại để so Flash-Lite vs Flash (mỗi model có thư mục cache riêng).